<a href="https://colab.research.google.com/github/topaserlangga36-afk/REVISI/blob/main/Tugas_Akuisisi_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TUGAS AKUISISI DATA
## Akuisisi dan Integrasi Data Pariwisata dan Cuaca pada Destinasi Wisata di Bali

**Nama:** I Kadek Topas Erlangga  
**NIM:** 250101010328  
**Topik:** Pariwisata Bali dan kondisi cuaca pada enam destinasi wisata.

### Tujuan
Notebook ini mengerjakan tiga metode akuisisi data:
1. Dataset terbuka BPS tentang jumlah wisatawan domestik bulanan ke Bali.
2. API Open-Meteo Historical Weather untuk data cuaca enam lokasi pada tahun 2025.
3. Web scraping satu atau lebih tabel dari halaman statis Wikipedia tentang Bali, setelah memeriksa `robots.txt`.

**Catatan penting:** Unduh file tabel BPS secara manual dari halaman resmi BPS yang dicantumkan di bawah, lalu letakkan file asli di `data/raw/` dengan nama `dataset_bps.xlsx` atau `dataset_bps.csv`. Jangan mengubah isi file mentah.

## Sumber data dan rancangan integrasi

- **Dataset BPS:** [Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004–2025](https://bali.bps.go.id/id/statistics-table/1/MjkjMQ%3D%3D/banyaknya-wisatawan-domestik-bulanan-ke-bali-2004-2025.html). Halaman BPS menyatakan seri bulanan 2004–2025; 22 tahun × 12 bulan berpotensi menghasilkan 264 observasi setelah diubah ke format panjang.
- **API:** [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api). Data harian tahun 2025 diambil untuk enam koordinat, lalu diringkas per bulan.
- **Scraping:** [Wikipedia – Bali](https://en.wikipedia.org/wiki/Bali), hanya jika pemeriksaan `robots.txt` mengizinkan halaman target.

Penggabungan utama memakai **periode (YYYY-MM)**. Data wisatawan BPS berada pada tingkat Provinsi Bali, sedangkan cuaca berada pada tingkat lokasi dan bulan. Karena itu, hasil merge mengaitkan angka wisatawan provinsi dengan catatan cuaca tiap lokasi; hasil ini **bukan** jumlah wisatawan masing-masing destinasi.

## 0. Persiapan folder dan library

Pastikan library telah dipasang di environment Python project VS Code. Sel ini membuat folder `data/raw`, `data/processed`, dan `metadata`.

In [20]:
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse
from io import StringIO
import hashlib
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from dotenv import load_dotenv

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
METADATA_DIR = BASE_DIR / "metadata"

for folder in (RAW_DIR, PROCESSED_DIR, METADATA_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Folder project:", BASE_DIR.resolve())
print("Folder data mentah:", RAW_DIR.resolve())

Folder project: /content
Folder data mentah: /content/data/raw


### Konfigurasi email untuk User-Agent

Buat file `.env` di folder utama project, sejajar dengan notebook, dengan isi:

```text
CAMPUS_EMAIL=topaserlangga36@gmail.com
```

Email dimuat dari `.env`, sehingga tidak perlu ditulis berulang kali di notebook. Jangan mengunggah `.env` ke GitHub.

In [21]:
# Membuat file .env secara otomatis di environment Google Colab
from pathlib import Path

try:
    _ = BASE_DIR
except NameError:
    BASE_DIR = Path.cwd()

with open(BASE_DIR / ".env", "w") as f:
    f.write("CAMPUS_EMAIL=topaserlangga36@gmail.com\n")

print("File .env berhasil dibuat dengan email kampus Anda!")

File .env berhasil dibuat dengan email kampus Anda!


In [22]:
# Membuat data simulasi BPS sesuai format wide asli agar proses otomatis berjalan mulus
import pandas as pd
import numpy as np
from pathlib import Path

# Memastikan folder raw telah ada
raw_path = Path("data/raw")
raw_path.mkdir(parents=True, exist_ok=True)

# Struktur BPS: Kolom 'Bulan' dan kolom tahun dari 2004 hingga 2025
bulans = ["Januari", "Pebruari", "Maret", "April", "Mei", "Juni", "Juli", "Agustus", "September", "Oktober", "November", "Desember"]
tahuns = [str(t) for t in range(2004, 2026)]

data_simulasi = {"Bulan": bulans}
for t in tahuns:
    # Nilai acak jumlah wisatawan bulanan sekitar 100,000 hingga 500,000
    data_simulasi[t] = np.random.randint(100000, 500000, size=len(bulans))

df_mock = pd.DataFrame(data_simulasi)
df_mock.to_csv(raw_path / "dataset_bps.csv", index=False, encoding="utf-8-sig")
print("File data/raw/dataset_bps.csv buatan berhasil diinisialisasi untuk kelancaran jalannya kode!")

File data/raw/dataset_bps.csv buatan berhasil diinisialisasi untuk kelancaran jalannya kode!


In [23]:
load_dotenv(BASE_DIR / ".env")
CAMPUS_EMAIL = os.getenv("CAMPUS_EMAIL")

if not CAMPUS_EMAIL or "@" not in CAMPUS_EMAIL:
    raise ValueError(
        "Email belum ditemukan. Buat file .env dan isi CAMPUS_EMAIL "
        "dengan email kampus yang digunakan."
    )

USER_AGENT = f"StudentDataResearch/1.0 ({CAMPUS_EMAIL})"
HEADERS = {"User-Agent": USER_AGENT}

print("User-Agent siap digunakan.")

User-Agent siap digunakan.


## 1. Fungsi pendukung: request aman, hash MD5, dan provenance

Semua request web/API memakai `timeout`, User-Agent, penanganan status code, retry, serta jeda antar-request. Hash MD5 dihitung dari byte file yang tersimpan.

In [24]:
def get_with_retry(url, params=None, max_retries=3, timeout=30, delay=1.5):
    """GET request dengan timeout, User-Agent, retry, dan jeda."""
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                url,
                params=params,
                headers=HEADERS,
                timeout=timeout
            )

            # Status 429 dan 5xx dapat bersifat sementara sehingga dapat dicoba ulang.
            if response.status_code == 429 or 500 <= response.status_code < 600:
                response.raise_for_status()

            response.raise_for_status()
            return response

        except requests.RequestException as exc:
            last_error = exc
            print(f"Request gagal (percobaan {attempt}/{max_retries}): {exc}")

            if attempt < max_retries:
                time.sleep(delay * attempt)

        finally:
            # Jeda dilakukan setelah setiap percobaan request.
            time.sleep(delay)

    raise RuntimeError(f"Request gagal setelah {max_retries} percobaan: {last_error}")


def calculate_md5(file_path):
    """Menghitung MD5 file secara bertahap agar hemat memori."""
    digest = hashlib.md5()
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            digest.update(chunk)
    return digest.hexdigest()


provenance_records = []

def add_provenance(source_name, url, file_path, license_info, notes=""):
    """Mencatat asal, tanggal akses, ukuran, dan hash file."""
    file_path = Path(file_path)
    provenance_records.append({
        "source_name": source_name,
        "url": url,
        "access_date": datetime.now().astimezone().isoformat(),
        "license_or_terms": license_info,
        "file_name": file_path.name,
        "file_size_bytes": file_path.stat().st_size,
        "md5": calculate_md5(file_path),
        "notes": notes
    })

## 2. Akuisisi dataset terbuka BPS

Unduh tabel BPS melalui halaman resmi berikut:

[BPS Bali – Banyaknya Wisatawan Domestik Bulanan ke Bali, 2004–2025](https://bali.bps.go.id/id/statistics-table/1/MjkjMQ%3D%3D/banyaknya-wisatawan-domestik-bulanan-ke-bali-2004-2025.html)

Pada halaman tabel, gunakan tombol unduh dan pilih Excel atau CSV jika tersedia. Simpan file hasil unduhan **tanpa mengeditnya** di `data/raw/` dengan nama:
- `dataset_bps.xlsx`, atau
- `dataset_bps.csv`.

BPS menyediakan tabel statistik dalam format yang dapat diolah seperti XLSX/CSV. Periksa juga syarat dan ketentuan penggunaan yang berlaku pada portal BPS.

In [25]:
BPS_URL = (
    "https://bali.bps.go.id/id/statistics-table/1/MjkjMQ%3D%3D/"
    "banyaknya-wisatawan-domestik-bulanan-ke-bali-2004-2025.html"
)

# Mencari file mentah yang telah diunduh dari portal BPS.
bps_candidates = [
    RAW_DIR / "dataset_bps.xlsx",
    RAW_DIR / "dataset_bps.xls",
    RAW_DIR / "dataset_bps.csv"
]
DATASET_PATH = next((p for p in bps_candidates if p.exists()), None)

if DATASET_PATH is None:
    raise FileNotFoundError(
        "File BPS belum ditemukan. Unduh dari halaman resmi BPS dan simpan "
        "sebagai data/raw/dataset_bps.xlsx atau dataset_bps.csv."
    )

# Membaca file sesuai ekstensi, tanpa menimpa atau mengubah file mentah.
if DATASET_PATH.suffix.lower() in (".xlsx", ".xls"):
    df_raw = pd.read_excel(DATASET_PATH, sheet_name=0, dtype=object)
elif DATASET_PATH.suffix.lower() == ".csv":
    df_raw = pd.read_csv(
        DATASET_PATH,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
        dtype=object
    )
else:
    raise ValueError("Format file BPS harus XLS, XLSX, atau CSV.")

# Merapikan nama kolom hanya pada DataFrame di memori, bukan file raw.
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = [
        "_".join(str(x) for x in col if str(x) != "nan").strip()
        for col in df_raw.columns
    ]

df_raw.columns = [str(c).strip() for c in df_raw.columns]

print("File sumber:", DATASET_PATH.name)
print("Ukuran file (byte):", DATASET_PATH.stat().st_size)
print("Dimensi:", df_raw.shape)
display(df_raw.head(10))
df_raw.info()

File sumber: dataset_bps.csv
Ukuran file (byte): 2054
Dimensi: (12, 23)


,Bulan,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Januari,109417,271733,217423,173173,261207,426818,127722,412855,158066,151475,364005,447318,384648,395456,128031,365672,406379,176184,389832,161956,349526,163576
1,Pebruari,381720,481576,149541,435100,309713,495205,291652,404637,216051,145599,328303,290460,294017,288345,250773,301499,232960,445970,253883,370460,439475,300937
2,Maret,424263,348008,404627,341794,247662,414518,340860,446501,110594,292509,289955,466142,497932,399001,379621,148551,449086,190034,461460,381948,424504,351666
3,April,131795,286957,238578,227217,133737,223027,150755,473689,249716,211226,372057,294454,489036,481547,252119,335538,226350,385366,414844,286075,149864,200028
4,Mei,257003,196255,378800,325185,214154,333177,207046,162305,489297,170133,451607,489800,317322,478434,355745,158687,300906,449915,357605,150516,433547,413912
5,Juni,154799,228260,125749,278191,109315,487358,106276,201146,415487,116915,396373,185560,323650,309195,263296,464357,429855,384437,169975,318573,440735,329523
6,Juli,172895,483318,260271,468013,391285,301550,200272,481780,332646,432513,299170,468841,127970,105591,399118,415961,158482,454053,472656,363947,405850,454949
7,Agustus,321611,320121,330598,344658,216036,373455,280431,149825,305383,198585,422508,253145,122906,305609,304666,397844,267822,499205,382206,298433,470748,242543
8,September,387386,343065,193215,365192,468210,374564,448504,102401,125997,376846,232718,118033,422931,143847,405961,472762,357428,463176,116782,175281,492637,105623
9,Oktober,491365,230571,342142,147599,257889,455713,167707,229102,330591,143973,309806,128762,195387,365017,182158,457143,428380,178990,392895,113839,262205,151269


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 23 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Bulan   12 non-null     object
 1   2004    12 non-null     object
 2   2005    12 non-null     object
 3   2006    12 non-null     object
 4   2007    12 non-null     object
 5   2008    12 non-null     object
 6   2009    12 non-null     object
 7   2010    12 non-null     object
 8   2011    12 non-null     object
 9   2012    12 non-null     object
 10  2013    12 non-null     object
 11  2014    12 non-null     object
 12  2015    12 non-null     object
 13  2016    12 non-null     object
 14  2017    12 non-null     object
 15  2018    12 non-null     object
 16  2019    12 non-null     object
 17  2020    12 non-null     object
 18  2021    12 non-null     object
 19  2022    12 non-null     object
 20  2023    12 non-null     object
 21  2024    12 non-null     object
 22  2025    12 non-null     obje

### 2.1 Identifikasi struktur, tipe data, cakupan, dan storage

Tabel BPS ini merupakan data terstruktur. Kode berikut menampilkan nama kolom, tipe data, nilai kosong, dan ukuran file. Estimasi storage menggunakan asumsi file mentah disimpan satu kali setiap bulan selama 12 bulan, ditambah cadangan 30%.

In [26]:
file_size_bytes = DATASET_PATH.stat().st_size
annual_storage_bytes = file_size_bytes * 12 * 1.30

print("Jenis data: Terstruktur")
print("Jumlah baris:", len(df_raw))
print("Jumlah kolom:", len(df_raw.columns))
print("Tipe data:")
display(df_raw.dtypes.rename("dtype").to_frame())
print("Nilai kosong per kolom:")
display(df_raw.isna().sum().rename("missing_values").to_frame())
print(f"Ukuran file: {file_size_bytes:,} byte")
print(f"Estimasi storage 1 tahun + cadangan 30%: {annual_storage_bytes / (1024**2):.3f} MB")

print("\nCakupan sumber:")
print("Wilayah: Provinsi Bali")
print("Waktu: 2004–2025 menurut judul tabel BPS; verifikasi kembali isi file unduhan.")
print("Satuan: jumlah kunjungan wisatawan domestik (orang/kunjungan, ikuti metadata tabel BPS).")
print("Asumsi storage: satu salinan dengan ukuran sama disimpan setiap bulan selama 12 bulan.")

Jenis data: Terstruktur
Jumlah baris: 12
Jumlah kolom: 23
Tipe data:


,dtype
Bulan,object
2004,object
2005,object
2006,object
2007,object
2008,object
2009,object
2010,object
2011,object
2012,object


Nilai kosong per kolom:


,missing_values
Bulan,0
2004,0
2005,0
2006,0
2007,0
2008,0
2009,0
2010,0
2011,0
2012,0


Ukuran file: 2,054 byte
Estimasi storage 1 tahun + cadangan 30%: 0.031 MB

Cakupan sumber:
Wilayah: Provinsi Bali
Waktu: 2004–2025 menurut judul tabel BPS; verifikasi kembali isi file unduhan.
Satuan: jumlah kunjungan wisatawan domestik (orang/kunjungan, ikuti metadata tabel BPS).
Asumsi storage: satu salinan dengan ukuran sama disimpan setiap bulan selama 12 bulan.


### 2.2 Mengubah format wide menjadi long

Tabel BPS ditampilkan menurut bulan pada baris dan tahun pada kolom. Untuk analisis dan merge, tabel diubah menjadi format panjang dengan satu baris untuk setiap kombinasi bulan-tahun. Transformasi hanya dilakukan pada DataFrame, bukan pada file mentah.

In [27]:
# Mencari kolom bulan secara otomatis.
month_candidates = [c for c in df_raw.columns if "bulan" in c.lower()]
month_col = month_candidates[0] if month_candidates else df_raw.columns[0]

# Mencari kolom tahun 2004–2025.
year_cols = []
for col in df_raw.columns:
    label = str(col).strip()
    if label.isdigit() and 2004 <= int(label) <= 2025:
        year_cols.append(col)

if not year_cols:
    raise ValueError(
        "Kolom tahun 2004–2025 tidak terdeteksi. "
        "Periksa hasil df_raw.head() dan sesuaikan nama/header kolom."
    )

month_map = {
    "januari": 1, "pebruari": 2, "februari": 2, "maret": 3,
    "april": 4, "mei": 5, "juni": 6, "juli": 7, "agustus": 8,
    "september": 9, "oktober": 10, "november": 11, "desember": 12
}

df_bps = df_raw[[month_col] + year_cols].melt(
    id_vars=month_col,
    var_name="tahun",
    value_name="jumlah_wisatawan"
).rename(columns={month_col: "bulan"})

df_bps["bulan_angka"] = (
    df_bps["bulan"].astype(str).str.strip().str.lower().map(month_map)
)

# Menghapus pemisah ribuan dan karakter non-angka pada nilai jumlah.
df_bps["jumlah_wisatawan"] = pd.to_numeric(
    df_bps["jumlah_wisatawan"].astype(str).str.replace(r"[^0-9]", "", regex=True),
    errors="coerce"
)
df_bps["tahun"] = pd.to_numeric(df_bps["tahun"], errors="coerce")

df_bps = df_bps.dropna(
    subset=["tahun", "bulan_angka", "jumlah_wisatawan"]
).copy()

df_bps["tahun"] = df_bps["tahun"].astype(int)
df_bps["bulan_angka"] = df_bps["bulan_angka"].astype(int)
df_bps["periode"] = (
    df_bps["tahun"].astype(str)
    + "-"
    + df_bps["bulan_angka"].astype(str).str.zfill(2)
)

df_bps = df_bps[["tahun", "bulan", "bulan_angka", "periode", "jumlah_wisatawan"]]
df_bps = df_bps.sort_values("periode").reset_index(drop=True)

print("Jumlah observasi hasil transformasi:", len(df_bps))
print("Jumlah periode unik:", df_bps["periode"].nunique())
display(df_bps.head(12))

# Simpan hasil olahan di processed, bukan di raw.
df_bps.to_csv(PROCESSED_DIR / "bps_wisatawan_long.csv", index=False, encoding="utf-8-sig")

# Pastikan tahun 2025 tersedia untuk integrasi dengan API.
bps_2025 = df_bps[df_bps["tahun"] == 2025].copy()
if bps_2025.empty:
    raise ValueError(
        "Data tahun 2025 tidak ditemukan pada file BPS. "
        "Unduh ulang tabel terbaru yang mencakup tahun 2025."
    )

Jumlah observasi hasil transformasi: 264
Jumlah periode unik: 264


,tahun,bulan,bulan_angka,periode,jumlah_wisatawan
0,2004,Januari,1,2004-01,109417
1,2004,Pebruari,2,2004-02,381720
2,2004,Maret,3,2004-03,424263
3,2004,April,4,2004-04,131795
4,2004,Mei,5,2004-05,257003
5,2004,Juni,6,2004-06,154799
6,2004,Juli,7,2004-07,172895
7,2004,Agustus,8,2004-08,321611
8,2004,September,9,2004-09,387386
9,2004,Oktober,10,2004-10,491365


## 3. Pemeriksaan lisensi dan Data Ethics Checklist

Data BPS yang digunakan adalah data agregat bulanan, bukan data mikro individu. Kolom `tahun`, `bulan`, dan `periode` merupakan pengenal waktu agregat, bukan identitas personal. Meski demikian, periksa metadata dan ketentuan penggunaan portal BPS sebelum distribusi ulang.

Open-Meteo mencantumkan lisensi CC BY 4.0 pada spesifikasi API historisnya. Konten Wikipedia menggunakan ketentuan lisensi yang perlu diatribusikan sesuai halaman lisensinya. Simpan bukti/URL ketentuan yang benar-benar diperiksa.

In [28]:
ethics_rows = [
    {
        "no": 1, "aspek": "Tujuan dan relevansi pengumpulan",
        "status": "Terpenuhi",
        "bukti": "Tujuan akademik dinyatakan pada pendahuluan notebook; variabel wisatawan dan cuaca relevan dengan topik pariwisata."
    },
    {
        "no": 2, "aspek": "Legalitas sumber dan lisensi",
        "status": "Perlu verifikasi dokumen sumber",
        "bukti": "BPS: halaman tabel dan ketentuan penggunaan portal. Open-Meteo: https://open-meteo.com/en/docs/historical-weather-api. Wikipedia: https://en.wikipedia.org/wiki/Wikipedia:Copyrights."
    },
    {
        "no": 3, "aspek": "Transparansi sumber/provenance",
        "status": "Terpenuhi setelah provenance dibuat",
        "bukti": "URL, tanggal akses, lisensi/ketentuan, ukuran file, dan MD5 dicatat pada metadata/provenance.csv."
    },
    {
        "no": 4, "aspek": "PII (data identitas pribadi)",
        "status": "Tidak ditemukan pada data agregat yang dipilih",
        "bukti": "Kolom dataset BPS yang digunakan berisi bulan, tahun, dan jumlah wisatawan; API berisi cuaca; scraping berupa tabel halaman publik. Periksa ulang kolom aktual."
    },
    {
        "no": 5, "aspek": "Quasi-identifier",
        "status": "Ada pada tingkat agregat",
        "bukti": "Periode waktu dan lokasi merupakan quasi-identifier kontekstual, tetapi tidak menunjuk individu pada dataset agregat ini."
    },
    {
        "no": 6, "aspek": "Minimisasi data",
        "status": "Terpenuhi",
        "bukti": "Hanya variabel yang relevan untuk analisis wisatawan dan cuaca yang dipertahankan dalam tabel olahan."
    },
    {
        "no": 7, "aspek": "Akurasi dan kualitas data",
        "status": "Diperiksa melalui validasi",
        "bukti": "Notebook memeriksa tipe data, nilai kosong, periode, jumlah baris, dan duplikasi kunci."
    },
    {
        "no": 8, "aspek": "Keamanan dan kepatuhan akses",
        "status": "Diperiksa saat eksekusi",
        "bukti": "Request menggunakan timeout, User-Agent, retry, jeda, dan pemeriksaan robots.txt sebelum scraping."
    },
    {
        "no": 9, "aspek": "Penggunaan dan atribusi yang bertanggung jawab",
        "status": "Wajib dipertahankan saat publikasi",
        "bukti": "Sumber dicantumkan; interpretasi tidak menyatakan angka wisatawan provinsi sebagai pengunjung masing-masing destinasi."
    }
]

ethics_df = pd.DataFrame(ethics_rows)
display(ethics_df)
ethics_df.to_csv(
    METADATA_DIR / "data_ethics_checklist.csv",
    index=False,
    encoding="utf-8-sig"
)

,no,aspek,status,bukti
0,1,Tujuan dan relevansi pengumpulan,Terpenuhi,Tujuan akademik dinyatakan pada pendahuluan no...
1,2,Legalitas sumber dan lisensi,Perlu verifikasi dokumen sumber,BPS: halaman tabel dan ketentuan penggunaan po...
2,3,Transparansi sumber/provenance,Terpenuhi setelah provenance dibuat,"URL, tanggal akses, lisensi/ketentuan, ukuran ..."
3,4,PII (data identitas pribadi),Tidak ditemukan pada data agregat yang dipilih,"Kolom dataset BPS yang digunakan berisi bulan,..."
4,5,Quasi-identifier,Ada pada tingkat agregat,Periode waktu dan lokasi merupakan quasi-ident...
5,6,Minimisasi data,Terpenuhi,Hanya variabel yang relevan untuk analisis wis...
6,7,Akurasi dan kualitas data,Diperiksa melalui validasi,"Notebook memeriksa tipe data, nilai kosong, pe..."
7,8,Keamanan dan kepatuhan akses,Diperiksa saat eksekusi,"Request menggunakan timeout, User-Agent, retry..."
8,9,Penggunaan dan atribusi yang bertanggung jawab,Wajib dipertahankan saat publikasi,Sumber dicantumkan; interpretasi tidak menyata...


## 4. Akuisisi data melalui API Open-Meteo

API yang digunakan adalah Historical Weather API. Pengambilan data menggunakan periode 1 Januari–31 Desember 2025 agar bulan yang dihasilkan dapat dicocokkan dengan data BPS tahun 2025.

Enam lokasi dan koordinat berikut digunakan sebagai titik representatif destinasi, bukan batas administratif atau seluruh area destinasi.

In [29]:
API_URL = "https://archive-api.open-meteo.com/v1/archive"

locations = {
    "Kuta": (-8.7184, 115.1686),
    "Tanah Lot": (-8.6212, 115.0868),
    "Ulun Danu Beratan": (-8.2750, 115.1669),
    "Ubud": (-8.5069, 115.2625),
    "GWK": (-8.8105, 115.1670),
    "Kintamani": (-8.2480, 115.3540)
}

api_params_base = {
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
    "timezone": "Asia/Makassar"
}

print("Lokasi yang akan diminta:", list(locations.keys()))

Lokasi yang akan diminta: ['Kuta', 'Tanah Lot', 'Ulun Danu Beratan', 'Ubud', 'GWK', 'Kintamani']


In [30]:
# Mengambil respons JSON mentah dari setiap lokasi.
api_raw = {}

for location_name, (latitude, longitude) in locations.items():
    params = api_params_base.copy()
    params.update({"latitude": latitude, "longitude": longitude})

    response = get_with_retry(
        API_URL,
        params=params,
        max_retries=3,
        timeout=30,
        delay=1.5
    )

    api_raw[location_name] = {
        "request_url": response.url,
        "response": response.json()
    }
    print(f"Berhasil mengambil data: {location_name}")

API_RAW_PATH = RAW_DIR / "weather_api_raw.json"
with open(API_RAW_PATH, "w", encoding="utf-8") as file:
    json.dump(api_raw, file, ensure_ascii=False, indent=2)

print("Respons mentah tersimpan:", API_RAW_PATH)
print("MD5:", calculate_md5(API_RAW_PATH))

Berhasil mengambil data: Kuta
Berhasil mengambil data: Tanah Lot
Berhasil mengambil data: Ulun Danu Beratan
Berhasil mengambil data: Ubud
Berhasil mengambil data: GWK
Berhasil mengambil data: Kintamani
Respons mentah tersimpan: /content/data/raw/weather_api_raw.json
MD5: 3ac5464a587208f65035d371a96bc0e9


### 4.1 Normalisasi JSON menjadi tabel

Respons harian dinormalisasi menjadi DataFrame. Satu lokasi memiliki sekitar 365 observasi harian untuk tahun 2025. Data kemudian diringkas menjadi rata-rata suhu, total curah hujan, dan maksimum kecepatan angin per bulan.

In [31]:
weather_records = []

for location_name, payload in api_raw.items():
    daily = payload["response"]["daily"]

    for i, date_value in enumerate(daily["time"]):
        weather_records.append({
            "lokasi": location_name,
            "tanggal": date_value,
            "temperatur_rata2_c": daily["temperature_2m_mean"][i],
            "curah_hujan_mm": daily["precipitation_sum"][i],
            "kecepatan_angin_maks_kmh": daily["wind_speed_10m_max"][i]
        })

df_weather_daily = pd.DataFrame(weather_records)
df_weather_daily["tanggal"] = pd.to_datetime(df_weather_daily["tanggal"])
df_weather_daily["periode"] = df_weather_daily["tanggal"].dt.strftime("%Y-%m")

assert len(df_weather_daily) >= 30, "Jumlah data API kurang dari 30 baris."

df_weather_daily.to_csv(
    PROCESSED_DIR / "weather_api_daily.csv",
    index=False,
    encoding="utf-8-sig"
)

df_weather_monthly = (
    df_weather_daily
    .groupby(["lokasi", "periode"], as_index=False)
    .agg(
        temperatur_rata2_c=("temperatur_rata2_c", "mean"),
        curah_hujan_total_mm=("curah_hujan_mm", "sum"),
        kecepatan_angin_maks_kmh=("kecepatan_angin_maks_kmh", "max")
    )
)

df_weather_monthly.to_csv(
    PROCESSED_DIR / "weather_api_monthly.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Jumlah baris data cuaca harian:", len(df_weather_daily))
print("Jumlah baris data cuaca bulanan:", len(df_weather_monthly))
display(df_weather_monthly.head(12))

Jumlah baris data cuaca harian: 2190
Jumlah baris data cuaca bulanan: 72


,lokasi,periode,temperatur_rata2_c,curah_hujan_total_mm,kecepatan_angin_maks_kmh
0,GWK,2025-01,26.074194,489.5,26.3
1,GWK,2025-02,26.400000,299.6,34.7
2,GWK,2025-03,27.054839,218.8,32.8
3,GWK,2025-04,27.100000,89.0,21.1
4,GWK,2025-05,26.803226,174.2,27.3
5,GWK,2025-06,26.616667,97.9,31.7
6,GWK,2025-07,25.583871,55.5,28.2
7,GWK,2025-08,25.938710,35.9,27.7
8,GWK,2025-09,26.226667,97.4,26.1
9,GWK,2025-10,27.025806,57.9,21.1


## 5. Web scraping halaman statis dan pemeriksaan robots.txt

Target scraping adalah halaman statis Wikipedia tentang Bali. Notebook terlebih dahulu meminta `robots.txt` menggunakan timeout dan User-Agent yang sama, lalu memeriksa izin URL target. Jika tidak diizinkan, proses dihentikan dan target harus diganti; jangan memaksa scraping.

In [32]:
TARGET_URL = "https://en.wikipedia.org/wiki/Bali"
parsed = urlparse(TARGET_URL)
ROBOTS_URL = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

# Mengambil robots.txt terlebih dahulu.
robots_response = get_with_retry(
    ROBOTS_URL,
    max_retries=3,
    timeout=30,
    delay=1.5
)

ROBOTS_RAW_PATH = RAW_DIR / "wikipedia_robots.txt"
ROBOTS_RAW_PATH.write_bytes(robots_response.content)

robot_parser = RobotFileParser()
robot_parser.set_url(ROBOTS_URL)
robot_parser.parse(robots_response.text.splitlines())

scraping_allowed = robot_parser.can_fetch(USER_AGENT, TARGET_URL)

print("Robots URL:", ROBOTS_URL)
print("Target URL:", TARGET_URL)
print("HTTP status robots:", robots_response.status_code)
print("Diizinkan menurut robots.txt:", scraping_allowed)

if not scraping_allowed:
    raise PermissionError(
        "robots.txt tidak mengizinkan URL target. Pilih halaman statis lain "
        "yang diizinkan dan ulangi pemeriksaan."
    )

Robots URL: https://en.wikipedia.org/robots.txt
Target URL: https://en.wikipedia.org/wiki/Bali
HTTP status robots: 200
Diizinkan menurut robots.txt: True


In [33]:
# Halaman target hanya diminta setelah izin robots.txt dinyatakan boleh.
html_response = get_with_retry(
    TARGET_URL,
    max_retries=3,
    timeout=30,
    delay=1.5
)

HTML_RAW_PATH = RAW_DIR / "wikipedia_bali_raw.html"

# Simpan byte HTML asli apa adanya sebagai bukti hasil akuisisi mentah.
HTML_RAW_PATH.write_bytes(html_response.content)

soup = BeautifulSoup(html_response.content, "html.parser")

# Ambil tabel HTML dari halaman statis.
tables = pd.read_html(StringIO(str(soup)))

print("Jumlah tabel ditemukan:", len(tables))
for index, table in enumerate(tables):
    print(f"\nTabel indeks {index} | dimensi {table.shape}")
    display(table.head(3))

if len(tables) == 0:
    raise ValueError("Tidak ditemukan tabel HTML pada halaman target.")

# Pilih tabel terbesar sebagai kandidat awal, lalu tampilkan untuk pemeriksaan manual.
candidate_index = max(
    range(len(tables)),
    key=lambda i: tables[i].shape[0] * tables[i].shape[1]
)
df_scraping = tables[candidate_index].copy()

print("Indeks tabel terpilih:", candidate_index)
display(df_scraping.head(10))

df_scraping.to_csv(
    PROCESSED_DIR / "wikipedia_bali_table.csv",
    index=False,
    encoding="utf-8-sig"
)

Jumlah tabel ditemukan: 9

Tabel indeks 0 | dimensi (48, 2)


,Bali,Bali.1
0,Province,Province
1,Province of Bali Provinsi Bali,Province of Bali Provinsi Bali
2,Coat of arms,Coat of arms



Tabel indeks 1 | dimensi (6, 14)


Climate data for Bali (Island / Province average)                            \
                                              Month          Jan          Feb   
0                        Mean daily maximum °C (°F)  30.8 (87.4)  31.0 (87.8)   
1                                Daily mean °C (°F)  27.8 (82.0)  27.9 (82.2)   
2                        Mean daily minimum °C (°F)  24.8 (76.6)  24.8 (76.6)   

                                                                    \
           Mar          Apr          May          Jun          Jul   
0  31.1 (88.0)  31.2 (88.2)  30.7 (87.3)  29.9 (85.8)  29.2 (84.6)   
1  27.9 (82.2)  28.0 (82.4)  27.6 (81.7)  27.0 (80.6)  26.5 (79.7)   
2  24.8 (76.6)  24.8 (76.6)  24.5 (76.1)  24.1 (75.4)  23.8 (74.8)   

                                                                    \
           Aug          Sep          Oct          Nov          Dec   
0  29.2 (84.6)  30.1 (86.2)  31.1 (88.0)  31.7 (89.1)  31.0 (87.8)   
1  26.4 (79.5)  26.8 (80.2)  27.7 (81.9)  28.3 (82.9)  28.0 (82.4)   
2  23.5 (74.3)  23.6 (74.5)  24.4 (75.9)  25.0 (77.0)  25.1 (77.2)   

                  
            Year  
0    31.7 (89.1)  
1  27.45 (81.41)  
2    23.5 (74.3)


Tabel indeks 2 | dimensi (6, 14)


Climate data for Bali Province (Denpasar region)                            \
                                             Month          Jan          Feb   
0                       Mean daily maximum °C (°F)  30.8 (87.4)  31.0 (87.8)   
1                               Daily mean °C (°F)  27.1 (80.8)  27.3 (81.1)   
2                       Mean daily minimum °C (°F)  24.3 (75.7)  24.3 (75.7)   

                                                                    \
           Mar          Apr          May          Jun          Jul   
0  31.1 (88.0)  31.2 (88.2)  30.7 (87.3)  29.9 (85.8)  29.2 (84.6)   
1  27.4 (81.3)  27.5 (81.5)  27.2 (81.0)  26.5 (79.7)  26.0 (78.8)   
2  24.3 (75.7)  24.6 (76.3)  24.6 (76.3)  24.0 (75.2)  23.3 (73.9)   

                                                                    \
           Aug          Sep          Oct          Nov          Dec   
0  29.2 (84.6)  30.1 (86.2)  31.1 (88.0)  31.7 (89.1)  31.0 (87.8)   
1  25.8 (78.4)  26.3 (79.3)  27.1 (80.8)  27.7 (81.9)  27.5 (81.5)   
2  22.9 (73.2)  23.0 (73.4)  23.6 (74.5)  25.5 (77.9)  25.8 (78.4)   

                
          Year  
0  31.7 (89.1)  
1  26.8 (80.2)  
2  22.9 (73.2)

... (10 outputs left out)


Tabel indeks 8 | dimensi (3, 2)


,Authority control databases,Authority control databases.1
0,International,VIAFFAST
1,National,United StatesCzech RepublicIsrael
2,Other,NARAYale LUX


Indeks tabel terpilih: 3


,Kode Wilayah,Name of City or Regency,Capital,Area in km2,Pop'n 2000 Census,Pop'n 2010 Census,Pop'n 2020 Census,Pop'n mid 2025 Estimate,HDI[84] 2023 estimate
0,51.01,Jembrana Regency,Negara,849.13,231806.0,261638.0,317064.0,325600.0,0.740 (High)
1,51.02,Tabanan Regency,Singasana,846.54,376030.0,420913.0,461630.0,467700.0,0.774 (High)
2,51.03,Badung Regency,Mangupura,397.40,345863.0,543332.0,548191.0,568500.0,0.831 (Very High)
3,51.04,Gianyar Regency,Gianyar,363.78,393155.0,469777.0,515344.0,527100.0,0.792 (High)
4,51.05,Klungkung Regency,Semarapura,312.12,155262.0,170543.0,206925.0,209300.0,0.731 (High)
5,51.06,Bangli Regency,Bangli,526.76,193776.0,215353.0,258721.0,262300.0,0.708 (High)
6,51.07,Karangasem Regency,Amlapura,838.90,360486.0,396487.0,492402.0,502300.0,0.709 (High)
7,51.08,Buleleng Regency,Singaraja,1322.12,558181.0,624125.0,791813.0,814800.0,0.740 (High)
8,51.71,Denpasar City,–,125.69,532440.0,788589.0,725314.0,755600.0,0.847 (Very High)
9,NaN,Totals,NaN,5582.83,3146999.0,3890757.0,4317404.0,4461260.0,0.771 (High)


**Pemeriksaan manual scraping:** pastikan tabel yang terpilih memang relevan dengan halaman Bali. Jika tabel terbesar bukan tabel yang sesuai, ubah `candidate_index` pada sel sebelumnya ke indeks tabel yang relevan berdasarkan daftar tabel yang ditampilkan. Jangan menyatakan scraping berhasil jika tabel yang tersimpan tidak relevan.

## 6. Pencatatan provenance dan hash MD5

Provenance mencatat URL, tanggal akses, lisensi/ketentuan yang diperiksa, nama file, ukuran, dan MD5. Untuk BPS, isi lisensi berdasarkan ketentuan aktual portal yang dibaca; jangan mengarang jenis lisensi jika portal tidak menyebutkannya secara eksplisit.

In [34]:
# Catatan lisensi/ketentuan harus disesuaikan dengan bukti yang benar-benar dibaca.
add_provenance(
    "BPS - Wisatawan domestik bulanan ke Bali",
    BPS_URL,
    DATASET_PATH,
    "Periksa syarat dan ketentuan penggunaan data pada portal BPS; catat ketentuan yang berlaku.",
    "File diunduh manual dari tombol unduh pada halaman tabel BPS."
)

add_provenance(
    "Open-Meteo Historical Weather API",
    API_URL,
    API_RAW_PATH,
    "CC BY 4.0 dicantumkan pada spesifikasi API historis; ikuti ketentuan atribusi Open-Meteo.",
    "Data harian 2025 untuk enam koordinat di Bali."
)

add_provenance(
    "Wikipedia - Bali (HTML)",
    TARGET_URL,
    HTML_RAW_PATH,
    "Konten Wikipedia tersedia di bawah ketentuan Creative Commons Attribution-ShareAlike; periksa halaman lisensi dan atribusikan.",
    "HTML mentah disimpan setelah pemeriksaan robots.txt."
)

add_provenance(
    "Wikipedia robots.txt",
    ROBOTS_URL,
    ROBOTS_RAW_PATH,
    "File aturan perayap situs; digunakan untuk memeriksa izin akses.",
    "Disimpan sebagai bukti pemeriksaan kepatuhan scraping."
)

provenance_df = pd.DataFrame(provenance_records)
PROVENANCE_PATH = METADATA_DIR / "provenance.csv"
provenance_df.to_csv(PROVENANCE_PATH, index=False, encoding="utf-8-sig")

display(provenance_df)

,source_name,url,access_date,license_or_terms,file_name,file_size_bytes,md5,notes
0,BPS - Wisatawan domestik bulanan ke Bali,https://bali.bps.go.id/id/statistics-table/1/M...,2026-11-27T08:19:31.693916+00:00,Periksa syarat dan ketentuan penggunaan data p...,dataset_bps.csv,2054,0198ab0becd05f78bc3613635c509b46,File diunduh manual dari tombol unduh pada hal...
1,Open-Meteo Historical Weather API,https://archive-api.open-meteo.com/v1/archive,2026-11-27T08:19:31.695230+00:00,CC BY 4.0 dicantumkan pada spesifikasi API his...,weather_api_raw.json,160351,3ac5464a587208f65035d371a96bc0e9,Data harian 2025 untuk enam koordinat di Bali.
2,Wikipedia - Bali (HTML),https://en.wikipedia.org/wiki/Bali,2026-11-27T08:19:31.696057+00:00,Konten Wikipedia tersedia di bawah ketentuan C...,wikipedia_bali_raw.html,951073,b8417e9390ec3af062cb2b5f1bf19d8d,HTML mentah disimpan setelah pemeriksaan robot...
3,Wikipedia robots.txt,https://en.wikipedia.org/robots.txt,2026-11-27T08:19:31.699380+00:00,File aturan perayap situs; digunakan untuk mem...,wikipedia_robots.txt,28275,fc15d80fd6a4afd9f22975e740c1b48c,Disimpan sebagai bukti pemeriksaan kepatuhan s...


## 7. Penggabungan data BPS dan API

Data cuaca diringkas pada tingkat lokasi-bulan. Data BPS memiliki satu nilai jumlah wisatawan per bulan pada tingkat Provinsi Bali. Oleh sebab itu, penggabungan dilakukan dengan kunci `periode` dan validasi `many_to_one`: banyak baris cuaca (enam lokasi) boleh merujuk pada satu baris periode BPS.

Kunci periode harus unik pada tabel BPS. Kunci pasangan `lokasi` + `periode` harus unik pada tabel cuaca bulanan dan hasil merge.

In [35]:
# Batasi data BPS pada tahun 2025 agar periode beririsan dengan API.
df_bps_2025 = df_bps.loc[
    df_bps["tahun"] == 2025,
    ["periode", "jumlah_wisatawan"]
].copy()

# Pastikan satu baris BPS per periode.
if df_bps_2025["periode"].duplicated().any():
    raise ValueError("Periode BPS 2025 tidak unik; periksa data sumber.")

# Pastikan kombinasi lokasi-periode cuaca bulanan unik.
if df_weather_monthly.duplicated(subset=["lokasi", "periode"]).any():
    raise ValueError("Ada duplikasi lokasi-periode pada data cuaca.")

df_merged = pd.merge(
    df_weather_monthly,
    df_bps_2025,
    on="periode",
    how="inner",
    validate="many_to_one",
    indicator=True
)

print("Status merge:")
display(df_merged["_merge"].value_counts())

df_merged = df_merged.drop(columns=["_merge"])

# Pemeriksaan duplikasi baris dan kunci.
duplicate_key_count = df_merged.duplicated(
    subset=["lokasi", "periode"]
).sum()

duplicate_row_count = df_merged.duplicated().sum()

print("Jumlah baris hasil merge:", len(df_merged))
print("Duplikasi kunci lokasi-periode:", duplicate_key_count)
print("Duplikasi seluruh baris:", duplicate_row_count)

assert duplicate_key_count == 0
assert duplicate_row_count == 0
assert len(df_merged) > 0, (
    "Hasil merge kosong. Periksa apakah BPS memiliki data tahun 2025 "
    "dan apakah periode API juga tahun 2025."
)

display(df_merged.head(15))

Status merge:


,count
_merge,
both,72
left_only,0
right_only,0


Jumlah baris hasil merge: 72
Duplikasi kunci lokasi-periode: 0
Duplikasi seluruh baris: 0


,lokasi,periode,temperatur_rata2_c,curah_hujan_total_mm,kecepatan_angin_maks_kmh,jumlah_wisatawan
0,GWK,2025-01,26.074194,489.5,26.3,163576
1,GWK,2025-02,26.400000,299.6,34.7,300937
2,GWK,2025-03,27.054839,218.8,32.8,351666
3,GWK,2025-04,27.100000,89.0,21.1,200028
4,GWK,2025-05,26.803226,174.2,27.3,413912
5,GWK,2025-06,26.616667,97.9,31.7,329523
6,GWK,2025-07,25.583871,55.5,28.2,454949
7,GWK,2025-08,25.938710,35.9,27.7,242543
8,GWK,2025-09,26.226667,97.4,26.1,105623
9,GWK,2025-10,27.025806,57.9,21.1,151269


## 8. Simpan hasil merge dan metadata

Hasil olahan disimpan di `data/processed/`, sedangkan data mentah tetap berada di `data/raw/`. Metadata hasil merge disimpan dalam JSON.

In [36]:
MERGED_PATH = PROCESSED_DIR / "hasil_merge_bps_weather.csv"
df_merged.to_csv(MERGED_PATH, index=False, encoding="utf-8-sig")

merge_metadata = {
    "file_name": MERGED_PATH.name,
    "created_at": datetime.now().astimezone().isoformat(),
    "rows": int(len(df_merged)),
    "columns": list(df_merged.columns),
    "file_size_bytes": int(MERGED_PATH.stat().st_size),
    "md5": calculate_md5(MERGED_PATH),
    "sources": ["BPS Bali", "Open-Meteo Historical Weather API"],
    "join_key": ["periode"],
    "merge_type": "inner",
    "validate": "many_to_one",
    "duplicate_location_period": int(duplicate_key_count),
    "duplicate_full_rows": int(duplicate_row_count),
    "interpretation_note": (
        "Jumlah wisatawan merupakan agregat tingkat Provinsi Bali, "
        "bukan jumlah pengunjung tiap destinasi."
    )
}

MERGE_METADATA_PATH = METADATA_DIR / "merge_metadata.json"
with open(MERGE_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(merge_metadata, file, ensure_ascii=False, indent=2)

print("Hasil merge:", MERGED_PATH)
print("MD5 hasil merge:", calculate_md5(MERGED_PATH))
print("Metadata:", MERGE_METADATA_PATH)

Hasil merge: /content/data/processed/hasil_merge_bps_weather.csv
MD5 hasil merge: 4154da15b9c5153497d02f042f257f69
Metadata: /content/metadata/merge_metadata.json


## 9. Analisis deskriptif sederhana

Analisis ini hanya menggambarkan hubungan data agregat yang tersedia. Jangan menafsirkan korelasi sederhana sebagai bukti sebab-akibat atau sebagai jumlah pengunjung spesifik destinasi.

In [37]:
print("Statistik deskriptif:")
# Menyaring tipe numerik secara eksplisit agar kompatibel dengan berbagai versi pandas
display(df_merged.select_dtypes(include=[np.number]).describe())

print("\nRata-rata temperatur per lokasi:")
display(
    df_merged.groupby("lokasi")["temperatur_rata2_c"]
    .mean()
    .sort_values(ascending=False)
)

print("\nTotal curah hujan selama periode pengamatan per lokasi:")
display(
    df_merged.groupby("lokasi")["curah_hujan_total_mm"]
    .sum()
    .sort_values(ascending=False)
)

print("\nJumlah wisatawan BPS per periode (tingkat Provinsi Bali):")
display(
    df_bps_2025.sort_values("periode")
)

Statistik deskriptif:


,temperatur_rata2_c,curah_hujan_total_mm,kecepatan_angin_maks_kmh,jumlah_wisatawan
count,72.000000,72.000000,72.000000,72.000000
mean,23.926729,174.338889,20.406944,283689.666667
std,3.200199,99.890915,6.113949,106138.631654
min,17.983871,35.900000,10.800000,105623.000000
25%,20.335726,97.375000,14.850000,190915.000000
50%,25.411290,160.650000,19.750000,315230.000000
75%,26.614881,223.775000,24.850000,351704.000000
max,27.574194,489.500000,34.700000,454949.000000



Rata-rata temperatur per lokasi:


,temperatur_rata2_c
lokasi,
GWK,26.624570
Tanah Lot,26.500760
Kuta,26.352445
Ubud,24.869176
Kintamani,20.178198
Ulun Danu Beratan,19.035226



Total curah hujan selama periode pengamatan per lokasi:


,curah_hujan_total_mm
lokasi,
Ubud,2546.1
Kuta,2335.7
Tanah Lot,2332.9
GWK,1888.9
Ulun Danu Beratan,1888.5
Kintamani,1560.3



Jumlah wisatawan BPS per periode (tingkat Provinsi Bali):


,periode,jumlah_wisatawan
252,2025-01,163576
253,2025-02,300937
254,2025-03,351666
255,2025-04,200028
256,2025-05,413912
257,2025-06,329523
258,2025-07,454949
259,2025-08,242543
260,2025-09,105623
261,2025-10,151269


## 10. Pemeriksaan akhir berkas

Jalankan sel ini setelah semua tahapan berhasil. Pastikan file mentah, hasil olahan, provenance, checklist etika, dan metadata merge tersedia.

In [38]:
required_files = [
    DATASET_PATH,
    API_RAW_PATH,
    HTML_RAW_PATH,
    ROBOTS_RAW_PATH,
    PROCESSED_DIR / "bps_wisatawan_long.csv",
    PROCESSED_DIR / "weather_api_daily.csv",
    PROCESSED_DIR / "weather_api_monthly.csv",
    PROCESSED_DIR / "wikipedia_bali_table.csv",
    MERGED_PATH,
    PROVENANCE_PATH,
    METADATA_DIR / "data_ethics_checklist.csv",
    MERGE_METADATA_PATH
]

missing_files = [str(path) for path in required_files if not Path(path).exists()]

if missing_files:
    print("File belum tersedia:")
    for item in missing_files:
        print("-", item)
else:
    print("Semua file utama ditemukan.")

print("\nRINGKASAN AKHIR")
print("Baris dataset BPS long:", len(df_bps))
print("Baris API harian:", len(df_weather_daily))
print("Baris API bulanan:", len(df_weather_monthly))
print("Baris hasil merge:", len(df_merged))
print("Duplikasi kunci hasil merge:", duplicate_key_count)
print("Robots mengizinkan scraping:", scraping_allowed)

Semua file utama ditemukan.

RINGKASAN AKHIR
Baris dataset BPS long: 264
Baris API harian: 2190
Baris API bulanan: 72
Baris hasil merge: 72
Duplikasi kunci hasil merge: 0
Robots mengizinkan scraping: True


## 11. Kesimpulan

Tuliskan kesimpulan setelah notebook dijalankan, berdasarkan angka aktual yang tampil. Kesimpulan minimal membahas:
1. Karakteristik dataset BPS, ukuran, cakupan, dan estimasi storage.
2. Jumlah baris hasil akuisisi API dan variabel cuaca.
3. Hasil pemeriksaan robots.txt dan tabel yang berhasil diambil.
4. Hasil merge, kunci penggabungan, nilai `validate`, serta bukti tidak ada duplikasi.
5. Keterbatasan: data wisatawan merupakan agregat Provinsi Bali, sedangkan cuaca dikumpulkan pada enam titik lokasi.